# LIANA analysis

**Developed by:** Anna Maguza\
**Affiliation:** Faculty of Medicine, Würzburg University\
**Creation date:** 10th March 2025\
**Last modified date:** 10th March 2025

## Import packages

In [2]:
import liana as li
import scanpy as sc
import pandas as pd
import numpy as np
from pycirclize import Circos
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex
from IPython.display import Image, display
from liana.mt import rank_aggregate
import os

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [1]:
!pip install pycirclize==1.7.1

  Attempting uninstall: pycirclize
    Found existing installation: pyCirclize 1.8.0
    Uninstalling pyCirclize-1.8.0:
      Successfully uninstalled pyCirclize-1.8.0

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


## Load Data

In [3]:
adata = sc.read_h5ad('data/gut_data/synthetic_spatial_43K_genes.h5ad')

In [4]:
adata

AnnData object with n_obs × n_vars = 274037 × 43704
    obs: 'Study_name', 'Donor_ID', 'Library_Preparation_Protocol', 'dataset', '_scvi_batch', '_scvi_labels', 'seed_labels', 'C_scANVI', 'SC_subsets', 'Cell_State', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_counts', 'REG4_score', 'gdT', 'Endothelial cells', 'latent_leiden_0.4', 'CD24_ligand_receptor_target_gene_GP', 'SLPI_ligand_receptor_target_gene_GP', 'CXCL14_ligand_receptor_target_gene_GP', 'ANPEP_ligand_receptor_target_gene_GP', 'IL1B_ligand_receptor_target_gene_GP', 'TIMP3_ligand_receptor_target_gene_GP', 'CDH1_ligand_receptor_target_gene_GP', 'TNXB_ligand_receptor_target_gene_GP', 'CLU_ligand_receptor_target_gene_GP', 'TFF1_ligand_receptor_target_gene_GP', 'CCL11_ligand_receptor_target_gene_GP', 'ROBO1_ligand_receptor_target_gene_GP', 'NRG3_ligand_recepto

## Normalize Data

In [5]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e6)
sc.pp.log1p(adata)

In [6]:
li.mt.rank_aggregate(adata,
                     groupby='C_scANVI',
                     resource_name='consensus',
                     expr_prop=0.1,
                     verbose=True, use_raw=False)

Using resource `consensus`.
Using `.X`!
Converting to sparse csr matrix!
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
14863 features of mat are empty, they will be removed.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:150: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/liana/method/_pipe_utils/_pre.py:153: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current b

Generating ligand-receptor stats for 274036 samples and 1401 features


/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/liana/method/sc/_liana_pipe.py:262: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:02<00:00, 15.89it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


In [7]:
adata.obs['C_scANVI'].value_counts()

C_scANVI
Myofibroblasts          41752
Fibroblasts             33793
Enterocyte              31889
Colonocyte              20706
Plasma cells            18561
Stem cells              14871
Macrophages             14387
Goblet cells            12620
B cells                 11914
CD4 T                   11635
Glial cells              7924
Endothelial cells        7098
TA                       6115
BEST4+ epithelial        5851
Pericytes                5828
Tuft cells               4808
EECs                     3786
Mast cells               3722
DC                       3409
arterial capillary       3242
Mature venous EC         2561
CD8 T                    2143
Mature arterial EC       1991
Monocytes                1733
LEC                       774
NK                        481
Immune Cycling cells      191
Lymphatics                 99
Microfold cell             95
Tregs                      28
ILCs                       18
gdT                         6
Adult Glia                  5
M

In [8]:
adata.uns['liana_res'].head()

## Traditional dot plot

In [9]:
li.pl.dotplot(adata = adata,
              colour='magnitude_rank',
              size='specificity_rank',
              inverse_size=True,
              inverse_colour=True,
              source_labels=['Stem cells'],
              target_labels=['Macrophages', 'Monocytes', 'LEC'],
              top_n=10,
              orderby='magnitude_rank',
              orderby_ascending=True,
              figure_size=(8, 7)
             )

## Custom circus plot

In [10]:
df = adata.uns['liana_res']

In [11]:
df['signalling'] = df['ligand_complex'].astype(str) + '-' + df['receptor_complex'].astype(str)

In [12]:
df

,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,expr_prod,scaled_weight,lr_logfc,spec_weight,lrscore,specificity_rank,magnitude_rank,signalling
12,Adult Glia,Adult Glia,C3,LRP1,7.978465,0.000,63.651112,1.924390,2.201129,0.061991,0.988356,0.067991,8.339706e-08,C3-LRP1
2892,Glial cells,Adult Glia,C3,LRP1,7.601396,0.000,57.686474,1.817804,2.677788,0.056182,0.987776,0.084911,1.334096e-06,C3-LRP1
8413,Pericytes,Macrophages,APP,CD74,7.614341,0.000,50.389023,1.541525,3.402609,0.011703,0.986932,0.128146,3.001331e-06,APP-CD74
8618,Pericytes,Monocytes,APP,CD74,7.612597,0.000,50.372063,1.541042,2.827166,0.011699,0.986930,0.190064,4.084882e-06,APP-CD74
4173,LEC,Macrophages,APP,CD74,7.453401,0.000,47.051369,1.488519,3.619842,0.010928,0.986483,0.153396,8.334889e-06,APP-CD74
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4356,LEC,Tregs,ADAM10,TSPAN14,1.021366,0.000,1.040682,0.621753,2.243976,0.005816,0.915638,1.000000,1.000000e+00,ADAM10-TSPAN14
3958,Immune Cycling cells,Macrophages,RPS19,C5AR1,1.222162,0.000,1.488882,0.869792,3.245121,0.018465,0.928481,0.084911,1.000000e+00,RPS19-C5AR1
4363,LEC,Tregs,ICAM1,IL2RA,1.086090,0.000,1.166416,0.741449,2.650308,0.013853,0.919941,0.110496,1.000000e+00,ICAM1-IL2RA
4110,LEC,Glial cells,ADAM10,AXL,1.016546,0.000,1.030355,0.587176,2.302300,0.015316,0.915252,0.363112,1.000000e+00,ADAM10-AXL


* there will be too much interactions, lets filter out what we really need

In [13]:
filtered_df = df[(df['magnitude_rank'] != 1)]

In [14]:
filtered_df.head()

,source,target,ligand_complex,receptor_complex,lr_means,cellphone_pvals,expr_prod,scaled_weight,lr_logfc,spec_weight,lrscore,specificity_rank,magnitude_rank,signalling
8413,Pericytes,Macrophages,APP,CD74,7.614341,0.0,50.389023,1.541525,3.402609,0.011703,0.986932,0.128146,0.000003,APP-CD74
8618,Pericytes,Monocytes,APP,CD74,7.612597,0.0,50.372063,1.541042,2.827166,0.011699,0.986930,0.190064,0.000004,APP-CD74
4173,LEC,Macrophages,APP,CD74,7.453401,0.0,47.051369,1.488519,3.619842,0.010928,0.986483,0.153396,0.000008,APP-CD74
4229,LEC,Monocytes,APP,CD74,7.451655,0.0,47.035534,1.488036,3.044400,0.010924,0.986481,0.231114,0.000010,APP-CD74
4634,Lymphatics,Macrophages,APP,CD74,7.448329,0.0,46.946201,1.486849,3.625854,0.010903,0.986468,0.154710,0.000012,APP-CD74


+ Filter cell states you are interested in

In [16]:
cell_types = ['Stem cells', 'Macrophages', 'B cells', 'Monocytes', 'arterial capillary', 'Tregs', 'Mature venous EC', 'Myofibroblasts', 'Mast cells', 'Pericytes', 'LEC', 'Endothelial cells', 'Mature arterial EC', 'CD4 T', 'Lymphatics', 'gdT']
filtered_df = df[df['source'].isin(cell_types)]
filtered_df = filtered_df[filtered_df['target'].isin(cell_types)]

In [ ]:
def create_circos_plot(matrix_df, filename, title):
    figures_dir = "/figures" ### PUT YOUR PATH HERE
    filepath = os.path.join(figures_dir, filename)
    
    filtered_df = matrix_df.loc[(matrix_df.sum(axis=1) != 0), (matrix_df.sum(axis=0) != 0)]
    
    if filtered_df.empty:
        print(f"No non-zero data to plot for {filepath}")
        return None
    
    circos = Circos.initialize_from_matrix(
        filtered_df,
        space=5,
        cmap="tab20c",
        label_kws=dict(size=8),
        link_kws=dict(alpha=0.7),
    )

    fig = circos.plotfig(figsize=(12, 12))
    plt.title(title, fontsize=16)
    fig.savefig(filepath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    
    print(f"Saved plot to {filepath}")
    return Image(filepath)

In [ ]:
signalling_pathways = filtered_df['signalling'].unique()

for pathway in signalling_pathways:
    pathway_df = filtered_df[filtered_df['signalling'] == pathway]
    
    matrix_df = pd.pivot_table(
        pathway_df, 
        values='magnitude_rank', 
        index='source', 
        columns='target', 
        fill_value=0
    )
    
    print(f"Creating circos plot for {pathway}")
    circos_image = create_circos_plot(
        matrix_df,
        f"{pathway}_circos.png", 
        f"{pathway} Interactions"
    )
    
    if circos_image:
        display(circos_image)